# ステップ1: データの読み込みと前処理

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# データの読み込み
file_path = './Egg_Production.csv'
egg_production_df = pd.read_csv(file_path)

# 最初の5行を表示
print(egg_production_df.head())

In [ ]:
# 欠損値のチェック
print(egg_production_df.isnull().sum())


In [ ]:
# 外れ値の確認
# 各特徴量について箱ひげ図を描く
plt.figure(figsize=(15, 10))

# 各特徴量の箱ひげ図を描画
for i, column in enumerate(egg_production_df.columns, 1):
    plt.subplot(2, 4, i)
    sns.boxplot(y=egg_production_df[column])
    plt.title(column)
    plt.ylabel('')

plt.tight_layout()
plt.show()

# 外れ値の除外
Q1 = egg_production_df.quantile(0.25)
Q3 = egg_production_df.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
egg_production_df_cleaned = egg_production_df[~((egg_production_df < lower_bound) | (egg_production_df > upper_bound)).any(axis=1)]


# ステップ2: 基本統計量とヒストグラム

In [ ]:
# 基本統計量の表示
egg_production_df_cleaned.describe().transpose()

In [ ]:
# 各データ項目のヒストグラムの描画
features = egg_production_df_cleaned.columns
plt.figure(figsize=(15, 12))

# 各データ項目の分布を確認 
for i, feature in enumerate(features, 1):
    plt.subplot(3, 3, i)
    egg_production_df_cleaned[feature].hist(bins=20, edgecolor='black',color='skyblue')
    plt.axvline(x=egg_production_df_cleaned[feature].mean(), color='red', linestyle='dashed', linewidth=2)
    plt.axvline(x=egg_production_df_cleaned[feature].mean() + egg_production_df_cleaned[feature].std(), color='green', linestyle='dashed', linewidth=2)
    plt.axvline(x=egg_production_df_cleaned[feature].mean() - egg_production_df_cleaned[feature].std(), color='green', linestyle='dashed', linewidth=2)
    plt.title(feature)

plt.tight_layout()
plt.show()

# ステップ3: 相関分析

In [ ]:
# 相関係数の計算と表示
correlation_matrix = egg_production_df_cleaned.corr()
correlation_with_target = correlation_matrix["Total_egg_production"].sort_values(ascending=False)
print(correlation_with_target)

# 相関行列の計算
correlation_matrix = egg_production_df_cleaned.corr()

# 相関ヒートマップの描画
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.show()

# 各特徴量と目的変数との相関係数を表示
target = 'Total_egg_production'
for feature in features:
    if feature != target:
        correlation = correlation_matrix[target][feature]
        evaluation = "弱い相関" if abs(correlation) < 0.3 else "正の相関" if correlation > 0 else "強い正の相関" if abs(correlation) > 0.7 else "弱い負の相関"
        print(f"{feature}と{target}の相関係数: {correlation:.2f} - {evaluation}")


# ステップ4: 単回帰モデルの構築と評価

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# 説明変数と目的変数の設定
X = egg_production_df_cleaned[['Amount_of_chicken']]
y = egg_production_df_cleaned['Total_egg_production']

# データの分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# モデルの構築
model = LinearRegression()
model.fit(X_train, y_train)

# 予測と評価
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print('Mean Squared Error:', mse)

In [ ]:
# 予測値の計算
X_values = egg_production_df_cleaned[['Amount_of_chicken']]
y_values_pred = model.predict(X_values)

# グラフ描画
plt.figure(figsize=(8, 6))
plt.scatter(egg_production_df_cleaned['Amount_of_chicken'], egg_production_df_cleaned['Total_egg_production'], label='Actual')
plt.plot(X_values, y_values_pred, color='red', label='Predicted')
plt.xlabel('Amount_of_chicken')
plt.ylabel('Total_egg_production')
plt.title('Prediction of Total Egg Production')
plt.legend()
plt.show()
